# FormPilot — train / eval on Colab

Runs `ml/train_lora.py` and `ml/eval.py` on a free T4 (fp16, ~10x the local GTX 1050 Ti).

**Before you start**
1. Locally: `python ml/colab_bundle.py` (add `--adapter checkpoints/<name>` to eval an adapter trained elsewhere).
2. Upload `data/colab/formpilot_colab.zip` to **My Drive/formpilot/**.
3. Here: *Runtime → Change runtime type → T4 GPU*, then run the cells top to bottom.

Checkpoints and predictions live on Drive, so after a disconnect just rerun every cell: training resumes from
the last checkpoint and eval skips examples it already predicted.

**After**: download `My Drive/formpilot/formpilot_out_<RUN>.zip`, then locally
`python ml/colab_bundle.py --unpack <zip>` and log the score with
`python ml/eval.py --preds data/runs/<RUN>_test.jsonl --name <RUN> --adapter checkpoints/<RUN> --prefill [--cot] --log`
(RESULTS.md is only ever written locally, against the bundled commit).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Unpack code (git bundle -> real commit) + dataset; put checkpoints/ and data/runs/ on Drive.
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/formpilot")
REPO = Path("/content/form-pilot")
BUNDLE = Path("/content/bundle")
if not REPO.exists():
    with zipfile.ZipFile(DRIVE / "formpilot_colab.zip") as z:
        z.extractall(BUNDLE)
    subprocess.run(["git", "clone", "-q", "-b", "main", str(BUNDLE / "repo.bundle"), str(REPO)], check=True)
    for d in [p.relative_to(BUNDLE) for p in (BUNDLE / "data").iterdir()]:  # dataset + its screenshot dirs
        shutil.move(BUNDLE / d, REPO / d)
    for d in ["checkpoints", "data/runs"]:
        (DRIVE / d).mkdir(parents=True, exist_ok=True)
        os.symlink(DRIVE / d, REPO / d)
    if (BUNDLE / "checkpoints").exists():  # adapter shipped for eval
        for a in (BUNDLE / "checkpoints").iterdir():
            if not (DRIVE / "checkpoints" / a.name).exists():
                shutil.copytree(a, DRIVE / "checkpoints" / a.name)
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r ml/requirements.txt

In [ ]:
RUN = "lora-cot"  # adapter -> My Drive/formpilot/checkpoints/<RUN>
COT = True  # with_cot targets (thought before action); False for no_cot
TRAIN = True  # False: only eval an adapter that came in the bundle
FLAG = "--cot" if COT else ""

In [ ]:
if TRAIN:
    !python ml/train_lora.py --name {RUN} {FLAG}

In [ ]:
!python ml/eval.py --adapter checkpoints/{RUN} --name {RUN} --prefill {FLAG}

In [ ]:
# Adapter (without checkpoint-* dirs) + test predictions -> one zip on Drive to download.
out = DRIVE / f"formpilot_out_{RUN}.zip"
with zipfile.ZipFile(out, "w") as z:
    for f in (REPO / "checkpoints" / RUN).iterdir():
        if f.is_file():
            z.write(f, f"checkpoints/{RUN}/{f.name}")
    z.write(REPO / "data" / "runs" / f"{RUN}_test.jsonl", f"data/runs/{RUN}_test.jsonl")
print(out)